# 01. 원본 CSV → 중복 제거·정렬된 Parquet

`datasets/archive/<DATASET>_Trans.csv` 를 읽어 money_laundry 추론서버(`binary_v10/preprocess.py`)가 기대하는
`tx` / `kyc` 형태의 Parquet로 만든다. 배치 버전은 같은 폴더의 `01_prepare_parquet.py`.

**규칙 (서비스 파이프라인과 맞춤)**
- 원본 헤더는 `Account` 열이 두 번 나와서 이름을 직접 준다 (위치 순서)
- 은행 코드 `"020"` → `20` (정수). 노드(계좌) 키 = `(bank_id, account)`
- `node_id` 는 `(bank_id, account)` 정렬 순서의 0..N-1 (v10 `materials.normalize` 와 같은 방식)
- 완전 중복 = 원본 11열이 모두 같은 행. 대표(edge_id 최소)만 남김. 대표의 `duplicate_group_id` 는 자기 edge_id, 단독 행은 NULL
  → v10 필터 `duplicate_group_id IS NULL OR edge_id = duplicate_group_id` 와 호환
- `edge_id` 는 `(ts, 11열)` 정렬 순서 → 파일 순서와 무관하게 재현 가능
- `ts` 는 원본 벽시계 그대로(타임존 없음). 서비스는 이걸 Asia/Seoul 로 읽는다

## 0. 설정

In [1]:
!pip install --break-system-packages --no-cache-dir -q duckdb pyarrow

In [2]:
from pathlib import Path
import json, re, time
import duckdb, pandas as pd

DATASET = "HI-Large"                 # HI-Small / HI-Medium / HI-Large / LI-*
THREADS, MEMORY = 4, "6GB"           # Large 는 8 / "24GB" 이상
WORK_DIR = None                      # 작업 duckdb 위치. None 이면 출력 폴더

# datasets 폴더를 현재 위치에서 위로 올라가며 찾는다 (노트북을 분석/ 에서 열든 루트에서 열든)
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "datasets").is_dir())
DATASETS = ROOT / "datasets"
OUT = ROOT / "parquets" / DATASET          # 출력: money_laundry_ai_/parquets/<DATASET>/
OUT.mkdir(parents=True, exist_ok=True)
FILES = {k: DATASETS / f"{DATASET}_{v}" for k, v in dict(trans="Trans.csv", accounts="accounts.csv", patterns="Patterns.txt").items()}
for p in FILES.values():
    assert p.is_file(), p
print(ROOT, "\n", OUT)

def q(v):                            # SQL 문자열 리터럴
    return "'" + str(v).replace("'", "''") + "'"

work = (Path(WORK_DIR) if WORK_DIR else OUT) / "work.duckdb"
for stale in (work, Path(str(work) + ".wal")):
    stale.unlink(missing_ok=True)
con = duckdb.connect(str(work))      # 파일 DB → 메모리 넘치면 디스크 스필
con.execute(f"SET threads={THREADS}"); con.execute(f"SET memory_limit='{MEMORY}'"); con.execute("SET preserve_insertion_order=false")
summary = {"dataset": DATASET}

/workspace 
 /workspace/분석/data/HI-Large


## 1. CSV 읽기

헤더를 무시(`header=false, skip=1`)하고 11개 이름을 직접 준다. 일단 전부 문자열(`all_varchar`)로 받고 다음 셀에서 형 변환 — 원본 값(`"020"` 같은 앞자리 0)을 그대로 보존해 두려는 것.

In [3]:
ELEVEN = ["ts_raw", "from_bank_raw", "from_account", "to_bank_raw", "to_account", "received_raw",
          "receiving_currency", "paid_raw", "payment_currency", "payment_format", "label_raw"]
names = lambda cols: "[" + ", ".join(f"'{c}'" for c in cols) + "]"

t = time.perf_counter()
con.execute(f"""
    CREATE OR REPLACE TABLE raw AS
    SELECT * FROM read_csv({q(FILES['trans'])}, header=false, skip=1, all_varchar=true, names={names(ELEVEN)}, parallel=true)
""")
summary["rows_raw"] = con.execute("SELECT count(*) FROM raw").fetchone()[0]
print(f"{summary['rows_raw']:,} 행  ({time.perf_counter()-t:.1f}s)")
con.execute("SELECT * FROM raw LIMIT 5").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

179,702,229 행  (195.7s)


,ts_raw,from_bank_raw,from_account,to_bank_raw,to_account,received_raw,receiving_currency,paid_raw,payment_currency,payment_format,label_raw
0,2022/08/01 00:03,0213868,818E238A0,0213868,818E238A0,3.75,US Dollar,3.75,US Dollar,Reinvestment,0
1,2022/08/01 00:14,0357345,818E236A0,0357345,818E236A0,1605.33,US Dollar,1605.33,US Dollar,Reinvestment,0
2,2022/08/01 00:01,0034458,818E23E30,0034458,818E23E30,102.00,US Dollar,102.00,US Dollar,Reinvestment,0
3,2022/08/01 00:28,0254115,818E23600,0254115,818E23600,10.84,US Dollar,10.84,US Dollar,Reinvestment,0
4,2022/08/01 00:04,0232913,818E24610,0232913,818E24610,6185.14,US Dollar,6185.14,US Dollar,Reinvestment,0


## 2. 형 변환 → 정렬 → `edge_id`

- `strptime('%Y/%m/%d %H:%M')` : 분 단위, 타임존 없음
- 금액은 `DOUBLE` (v10 전처리가 float64 로 받음)
- `edge_id = row_number() OVER (ORDER BY ts, 11열)` : 완전 중복 행은 인접하게 되고, 재실행해도 같은 번호

In [4]:
t = time.perf_counter()
con.execute("""
    CREATE OR REPLACE TABLE typed AS
    SELECT strptime(ts_raw, '%Y/%m/%d %H:%M') AS ts,
           CAST(from_bank_raw AS BIGINT) AS from_bank, from_account,
           CAST(to_bank_raw AS BIGINT)   AS to_bank,   to_account,
           CAST(paid_raw AS DOUBLE) AS paid, CAST(received_raw AS DOUBLE) AS received,
           payment_currency, receiving_currency, payment_format,
           CAST(label_raw AS INTEGER) = 1 AS is_laundering,
           """ + ", ".join(ELEVEN) + """
    FROM raw""")
con.execute("DROP TABLE raw")
con.execute("""
    CREATE OR REPLACE TABLE ordered AS
    SELECT row_number() OVER (ORDER BY ts, from_bank, from_account, to_bank, to_account, paid, received,
                                       payment_currency, receiving_currency, payment_format, is_laundering) AS edge_id, *
    FROM typed""")
con.execute("DROP TABLE typed")
print(f"{time.perf_counter()-t:.1f}s")
con.execute("SELECT edge_id, ts, from_bank, from_account, to_bank, to_account, paid, payment_currency, payment_format, is_laundering FROM ordered ORDER BY edge_id LIMIT 5").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2424.7s


,edge_id,ts,from_bank,from_account,to_bank,to_account,paid,payment_currency,payment_format,is_laundering
0,1,2022-08-01,0,80006DFE0,0,80006DFE0,19.18,US Dollar,Reinvestment,False
1,2,2022-08-01,0,800077B00,867,80079A2C0,17955.11,US Dollar,Cheque,False
2,3,2022-08-01,0,80010E900,1208,80011CC20,1199.83,US Dollar,Cash,False
3,4,2022-08-01,0,800117F00,16315,8040DF950,148565.46,US Dollar,Wire,False
4,5,2022-08-01,0,8001202E0,20,800226BD0,227247.42,US Dollar,Cheque,False


## 3. 완전 중복 탐지

원본 11열 전부로 파티션을 잡아 그룹의 최소 `edge_id`(= 대표)와 크기를 붙인다.
서비스의 통합 단계(`report_matching`)는 반복 동일 행을 *발생 건수대로 별도 거래*로 보존하지만,
v10 전처리는 대표 1건만 쓴다 — 여기선 모델 쪽 규칙을 따른다.

In [5]:
key = ", ".join(ELEVEN)
con.execute(f"""
    CREATE OR REPLACE TABLE grouped AS
    SELECT edge_id, min(edge_id) OVER (PARTITION BY {key}) AS rep_id, count(*) OVER (PARTITION BY {key}) AS group_size
    FROM ordered""")
summary["rows_duplicate_removed"] = con.execute("SELECT count(*) FROM grouped WHERE edge_id <> rep_id").fetchone()[0]
summary["duplicate_groups"] = con.execute("SELECT count(DISTINCT rep_id) FROM grouped WHERE group_size > 1").fetchone()[0]
print(f"중복 그룹 {summary['duplicate_groups']:,}개, 제거될 행 {summary['rows_duplicate_removed']:,}")
# 중복 그룹 실제 내용 확인
con.execute("""SELECT g.rep_id, g.group_size, o.* FROM grouped g JOIN ordered o USING (edge_id)
               WHERE g.group_size > 1 ORDER BY g.rep_id, o.edge_id LIMIT 10""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

중복 그룹 150개, 제거될 행 150


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rep_id,group_size,edge_id,ts,from_bank,from_account,to_bank,to_account,paid,received,...,from_bank_raw,from_account_1,to_bank_raw,to_account_1,received_raw,receiving_currency_1,paid_raw,payment_currency_1,payment_format_1,label_raw
0,5450308,2,5450308,2022-08-02 11:49:00,12425,803AD6CA1,2219870,850BF6F41,0.000001,0.000001,...,012425,803AD6CA1,02219870,850BF6F41,0.000001,Bitcoin,0.000001,Bitcoin,Bitcoin,0
1,5450308,2,5450309,2022-08-02 11:49:00,12425,803AD6CA1,2219870,850BF6F41,0.000001,0.000001,...,012425,803AD6CA1,02219870,850BF6F41,0.000001,Bitcoin,0.000001,Bitcoin,Bitcoin,0
2,8695746,2,8695746,2022-08-04 03:58:00,2156357,84DD03B71,214266,85025CAF1,0.000001,0.000001,...,02156357,84DD03B71,00214266,85025CAF1,0.000001,Bitcoin,0.000001,Bitcoin,Bitcoin,0
3,8695746,2,8695747,2022-08-04 03:58:00,2156357,84DD03B71,214266,85025CAF1,0.000001,0.000001,...,02156357,84DD03B71,00214266,85025CAF1,0.000001,Bitcoin,0.000001,Bitcoin,Bitcoin,0
4,8695748,2,8695748,2022-08-04 03:58:00,2156357,84DD03B71,2156357,84DD03B70,0.010000,0.000001,...,02156357,84DD03B71,02156357,84DD03B70,0.000001,Bitcoin,0.01,US Dollar,ACH,0
5,8695748,2,8695749,2022-08-04 03:58:00,2156357,84DD03B71,2156357,84DD03B70,0.010000,0.000001,...,02156357,84DD03B71,02156357,84DD03B70,0.000001,Bitcoin,0.01,US Dollar,ACH,0
6,8791620,2,8791620,2022-08-04 05:12:00,266092,818474C81,2216852,851A5BA11,0.000005,0.000005,...,0266092,818474C81,02216852,851A5BA11,0.000005,Bitcoin,0.000005,Bitcoin,Bitcoin,0
7,8791620,2,8791621,2022-08-04 05:12:00,266092,818474C81,2216852,851A5BA11,0.000005,0.000005,...,0266092,818474C81,02216852,851A5BA11,0.000005,Bitcoin,0.000005,Bitcoin,Bitcoin,0
8,13378100,2,13378100,2022-08-06 01:22:00,122407,80E54B721,213589,84EC258A1,0.000002,0.000002,...,0122407,80E54B721,00213589,84EC258A1,0.000002,Bitcoin,0.000002,Bitcoin,Bitcoin,0
9,13378100,2,13378101,2022-08-06 01:22:00,122407,80E54B721,213589,84EC258A1,0.000002,0.000002,...,0122407,80E54B721,00213589,84EC258A1,0.000002,Bitcoin,0.000002,Bitcoin,Bitcoin,0


## 4. 노드(계좌) 표 `kyc`

거래에 등장한 `(bank_id, account)` 전부를 노드로 삼고, `accounts.csv` 를 LEFT JOIN 해서 소유주(Entity)·이름·은행명을 붙인다.
서비스의 `entity_attributes.py` 와 같은 규칙으로 이름에서 범주만 뽑는다:
- `entity_type`: `"Corporation #33520"` → `Corporation` (정규식 `^(.*\S) #\d+$`)
- `country`: `"Canada Bank #27"` → `Canada` (정규식 `^(.*?) Bank #`)

In [6]:
ACCOUNT_COLUMNS = ["bank_name", "bank_id_raw", "account", "entity_id", "entity_name"]
con.execute(f"""
    CREATE OR REPLACE TABLE acct AS
    SELECT CAST(bank_id_raw AS BIGINT) AS bank_id, account, entity_id, entity_name, bank_name
    FROM read_csv({q(FILES['accounts'])}, header=false, skip=1, all_varchar=true, names={names(ACCOUNT_COLUMNS)})""")
con.execute("""
    CREATE OR REPLACE TABLE kyc AS
    WITH parties AS (SELECT from_bank AS bank_id, from_account AS account FROM ordered
                     UNION SELECT to_bank, to_account FROM ordered),
    joined AS (
      SELECT p.bank_id, p.account, a.entity_id, a.entity_name, a.bank_name,
             regexp_extract(a.entity_name, '^(.*\\S) #\\d+$', 1) AS entity_type_raw,
             regexp_extract(a.bank_name,   '^(.*?) Bank #', 1)   AS country_raw
      FROM parties p LEFT JOIN acct a USING (bank_id, account))
    SELECT row_number() OVER (ORDER BY bank_id, account) - 1 AS node_id,
           bank_id, account, entity_id, entity_name, bank_name,
           nullif(entity_type_raw, '') AS entity_type, nullif(country_raw, '') AS country
    FROM joined""")
summary["nodes"] = con.execute("SELECT count(*) FROM kyc").fetchone()[0]
summary["nodes_without_account_row"] = con.execute("SELECT count(*) FROM kyc WHERE entity_id IS NULL").fetchone()[0]
print(f"노드 {summary['nodes']:,} (accounts.csv 에 없는 계좌 {summary['nodes_without_account_row']:,})")
display(con.execute("SELECT entity_type, count(*) n FROM kyc GROUP BY 1 ORDER BY n DESC").df())
con.execute("SELECT * FROM kyc LIMIT 5").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

노드 2,116,168 (accounts.csv 에 없는 계좌 0)


,entity_type,n
0,Partnership,741335
1,Corporation,689899
2,Sole Proprietorship,676146
3,Country,5689
4,Individual,3071
5,Direct,28


,node_id,bank_id,account,entity_id,entity_name,bank_name,entity_type,country
0,0,0,8000474C0,2AA02E7E5F0,Corporation #114,Hearthstone Bancorp,Corporation,None
1,1,0,800047930,2AA02E7E650,Partnership #152,Hearthstone Bancorp,Partnership,None
2,2,0,80006C140,2AA02EA9B10,Corporation #68,Hearthstone Bancorp,Corporation,None
3,3,0,80006DFE0,2AA02E7E6B0,Sole Proprietorship #131,Hearthstone Bancorp,Sole Proprietorship,None
4,4,0,80006F150,2AA02E8D020,Sole Proprietorship #130,Hearthstone Bancorp,Sole Proprietorship,None


## 5. `transactions.parquet` / `duplicates.parquet` / `kyc.parquet`

`src`/`dst` 를 `node_id` 로 바꾸고, 중복 대표만 남겨 `ts, edge_id` 순으로 쓴다. 제거된 행은 `duplicates.parquet` 에 따로.

In [1]:
con.execute("""
    CREATE OR REPLACE VIEW tx_all AS
    SELECT o.edge_id, o.ts, s.node_id AS src, d.node_id AS dst,
           o.paid, o.received, o.payment_currency, o.receiving_currency, o.payment_format,
           CASE WHEN g.group_size > 1 THEN g.rep_id END AS duplicate_group_id,
           o.is_laundering, o.from_bank, o.from_account, o.to_bank, o.to_account, CAST(o.ts AS DATE) AS day, g.group_size
    FROM ordered o JOIN grouped g USING (edge_id)
    JOIN kyc s ON s.bank_id = o.from_bank AND s.account = o.from_account
    JOIN kyc d ON d.bank_id = o.to_bank   AND d.account = o.to_account""")
t = time.perf_counter()
con.execute(f"""COPY (SELECT * EXCLUDE (group_size) FROM tx_all WHERE duplicate_group_id IS NULL OR edge_id = duplicate_group_id ORDER BY ts, edge_id)
                TO {q(OUT / 'transactions.parquet')} (FORMAT PARQUET, COMPRESSION ZSTD)""")
con.execute(f"""COPY (SELECT * FROM tx_all WHERE edge_id <> duplicate_group_id ORDER BY duplicate_group_id, edge_id)
                TO {q(OUT / 'duplicates.parquet')} (FORMAT PARQUET, COMPRESSION ZSTD)""")
con.execute(f"COPY (SELECT * FROM kyc ORDER BY node_id) TO {q(OUT / 'kyc.parquet')} (FORMAT PARQUET, COMPRESSION ZSTD)")
row = con.execute("""SELECT count(*), min(ts), max(ts), sum(is_laundering::INT), count(*) FILTER (WHERE src = dst)
                     FROM tx_all WHERE duplicate_group_id IS NULL OR edge_id = duplicate_group_id""").fetchone()
summary.update(rows_kept=row[0], ts_min=str(row[1]), ts_max=str(row[2]), laundering_rows=row[3], self_transfers=row[4])
print(f"남은 거래 {row[0]:,}  기간 {row[1]} ~ {row[2]}  라벨1 {row[3]:,}  셀프이체 {row[4]:,}   ({time.perf_counter()-t:.1f}s)")
pd.read_parquet(OUT / "transactions.parquet").head()

NameError: name 'con' is not defined

## 6. `Patterns.txt` → `patterns.parquet`

```
BEGIN LAUNDERING ATTEMPT - FAN-OUT:  Max 16-degree Fan-Out
2022/09/01 00:06,021174,800737690,012,80011F990,2848.96,Euro,2848.96,Euro,ACH,1
...
END LAUNDERING ATTEMPT - FAN-OUT
```
시도(attempt)마다 번호를 붙이고, 각 행을 11열 전부로 거래와 매칭해 `edge_id` 를 얻는다. 라벨1 거래 중 패턴 파일에 없는 것이 "unmatched laundering".

In [3]:
import duckdb
con = duckdb.connect()
for name, expect in [("transactions.parquet", 179_702_079), ("duplicates.parquet", 150), ("kyc.parquet", 2_116_168)]:
    p = OUT / name
    try:
        n, = con.execute(f"SELECT count(*) FROM read_parquet({q(p)})").fetchone()
        groups = con.execute(f"SELECT count(*) FROM parquet_metadata({q(p)})").fetchone()[0]
        print(f"{name:22s} {n:>13,} 행  (기대 {expect:,}) {'OK' if n == expect else '불일치'}   row groups {groups}  {p.stat().st_size/1e9:.2f} GB")
    except Exception as e:
        print(f"{name:22s} 열기 실패 → 불완전: {e}")
# 정렬 확인 (ts 단조증가) — 전체 스캔이라 Large 는 1~2분
print("정렬 OK:", con.execute(f"""SELECT bool_and(ok) FROM (SELECT ts >= lag(ts) OVER (ORDER BY edge_id) AS ok
                                      FROM read_parquet({q(OUT/'transactions.parquet')})) WHERE ok IS NOT NULL""").fetchone()[0])

transactions.parquet     179,702,079 행  (기대 179,702,079) OK   row groups 23408  4.80 GB
duplicates.parquet               150 행  (기대 150) OK   row groups 17  0.00 GB
kyc.parquet                2,116,168 행  (기대 2,116,168) OK   row groups 144  0.03 GB


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

정렬 OK: True


In [6]:
# 6. Patterns.txt → patterns.parquet
#   - BEGIN 줄에 콜론이 없는 타입(BIPARTITE, SCATTER-GATHER, STACK)도 잡도록 정규식 수정
#   - tx_all 뷰 대신 이미 쓴 transactions.parquet 에서 매칭 (재계산 없음, 커널 재시작 후 0번 셀만 돌리고 실행 가능)
_BEGIN = re.compile(r"^BEGIN LAUNDERING ATTEMPT - ([A-Z\- ]+?)(?::\s*(.*))?$")
tmp = work.parent / f"_patterns_{DATASET}.tsv"
attempt, current, orphan = 0, None, 0
with FILES["patterns"].open(encoding="utf-8") as src, tmp.open("w", encoding="utf-8", newline="") as dst:
    for line in src:
        line = line.rstrip("\r\n")
        if not line:
            continue
        m = _BEGIN.match(line)
        if m:
            attempt += 1
            current = (attempt, m.group(1).strip(), (m.group(2) or "").strip())
            continue
        if line.startswith("END LAUNDERING ATTEMPT"):
            current = None
            continue
        if current is None:
            orphan += 1            # BEGIN 없이 나온 행 — 0 이어야 정상
            continue
        dst.write(f"{current[0]}\t{current[1]}\t{current[2]}\t{line}\n")
print(f"시도 {attempt:,}개 파싱, BEGIN 밖의 행 {orphan}")

con.execute(f"""
    CREATE OR REPLACE TABLE pat AS
    WITH r AS (SELECT attempt_id, pattern_type, pattern_desc, str_split(row, ',') AS f
               FROM read_csv({q(tmp)}, header=false, delim='\t', quote='', all_varchar=true,
                             names=['attempt_id','pattern_type','pattern_desc','row']))
    SELECT CAST(attempt_id AS INTEGER) AS attempt_id, pattern_type, pattern_desc,
           strptime(f[1], '%Y/%m/%d %H:%M') AS ts, CAST(f[2] AS BIGINT) AS from_bank, f[3] AS from_account,
           CAST(f[4] AS BIGINT) AS to_bank, f[5] AS to_account, CAST(f[6] AS DOUBLE) AS received, f[7] AS receiving_currency,
           CAST(f[8] AS DOUBLE) AS paid, f[9] AS payment_currency, f[10] AS payment_format
    FROM r WHERE len(f) = 11""")
display(con.execute("SELECT pattern_type, count(DISTINCT attempt_id) AS attempts, count(*) AS n_rows FROM pat GROUP BY 1 ORDER BY 1").df())

t = time.perf_counter()
con.execute(f"""
    COPY (SELECT p.attempt_id, p.pattern_type, p.pattern_desc, t.edge_id, p.ts,
                 count(*) OVER (PARTITION BY p.attempt_id) AS attempt_size, t.edge_id IS NULL AS unmatched
          FROM pat p
          LEFT JOIN read_parquet({q(OUT / 'transactions.parquet')}) t
            ON t.ts = p.ts AND t.from_bank = p.from_bank AND t.from_account = p.from_account AND t.to_bank = p.to_bank
           AND t.to_account = p.to_account AND t.paid = p.paid AND t.received = p.received
           AND t.payment_currency = p.payment_currency AND t.receiving_currency = p.receiving_currency AND t.payment_format = p.payment_format
          ORDER BY p.attempt_id, p.ts, t.edge_id)
    TO {q(OUT / 'patterns.parquet')} (FORMAT PARQUET, COMPRESSION ZSTD)""")
tmp.unlink(missing_ok=True)

pat = pd.read_parquet(OUT / "patterns.parquet")
summary.update(pattern_attempts=attempt, pattern_rows=len(pat), pattern_rows_unmatched=int(pat.unmatched.sum()),
               pattern_types=sorted(pat.pattern_type.unique()))
print(f"패턴 행 {len(pat):,} (거래와 매칭 안 된 행 {int(pat.unmatched.sum()):,})  ({time.perf_counter()-t:.0f}s)")
display(pat.groupby("pattern_type").agg(attempts=("attempt_id", "nunique"), rows=("edge_id", "size"), unmatched=("unmatched", "sum")))
pat.head()

시도 16,467개 파싱, BEGIN 밖의 행 0


,pattern_type,attempts,n_rows
0,BIPARTITE,2109,12174
1,CYCLE,2086,12340
2,FAN-IN,2014,13287
3,FAN-OUT,2040,13554
4,GATHER-SCATTER,2054,26361
5,RANDOM,2001,9618
6,SCATTER-GATHER,2067,26978
7,STACK,2096,23624


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

패턴 행 137,936 (거래와 매칭 안 된 행 0)  (12s)


,attempts,rows,unmatched
pattern_type,,,
BIPARTITE,2109,12174,0
CYCLE,2086,12340,0
FAN-IN,2014,13287,0
FAN-OUT,2040,13554,0
GATHER-SCATTER,2054,26361,0
RANDOM,2001,9618,0
SCATTER-GATHER,2067,26978,0
STACK,2096,23624,0


,attempt_id,pattern_type,pattern_desc,edge_id,ts,attempt_size,unmatched
0,1,STACK,NaN,17329971,2022-08-09 05:14:00,6,False
1,1,STACK,NaN,25736114,2022-08-13 12:40:00,6,False
2,1,STACK,NaN,25751590,2022-08-13 13:09:00,6,False
3,1,STACK,NaN,27585927,2022-08-15 07:40:00,6,False
4,1,STACK,NaN,28104526,2022-08-15 14:19:00,6,False


## 7. 마무리 — `summary.json`, 작업 파일 정리

In [7]:
# 7. 마무리 — summary.json, 작업 파일 정리
# (5번 셀에서 커널이 죽었어도 parquet 에서 수치를 다시 읽어 summary 를 채운다)
row = con.execute(f"""SELECT count(*), min(ts), max(ts), sum(is_laundering::INT), count(*) FILTER (WHERE src = dst)
                      FROM read_parquet({q(OUT / 'transactions.parquet')})""").fetchone()
summary.update(
    rows_kept=row[0], ts_min=str(row[1]), ts_max=str(row[2]), laundering_rows=row[3], self_transfers=row[4],
    rows_duplicate_removed=con.execute(f"SELECT count(*) FROM read_parquet({q(OUT / 'duplicates.parquet')})").fetchone()[0],
    duplicate_groups=con.execute(f"SELECT count(DISTINCT duplicate_group_id) FROM read_parquet({q(OUT / 'duplicates.parquet')})").fetchone()[0],
    nodes=con.execute(f"SELECT count(*) FROM read_parquet({q(OUT / 'kyc.parquet')})").fetchone()[0],
    nodes_without_account_row=con.execute(f"SELECT count(*) FROM read_parquet({q(OUT / 'kyc.parquet')}) WHERE entity_id IS NULL").fetchone()[0],
)
summary.setdefault("rows_raw", summary["rows_kept"] + summary["rows_duplicate_removed"])

con.close()
for stale in (work, Path(str(work) + ".wal")):
    stale.unlink(missing_ok=True)
(OUT / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print(json.dumps(summary, ensure_ascii=False, indent=2, default=str))
for f in sorted(OUT.glob("*.parquet")) + [OUT / "summary.json"]:
    print(f"{f.name:22s} {f.stat().st_size/1e6:9.1f} MB")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

{
  "dataset": "HI-Large",
  "pattern_attempts": 16467,
  "pattern_rows": 137936,
  "pattern_rows_unmatched": 0,
  "pattern_types": [
    "BIPARTITE",
    "CYCLE",
    "FAN-IN",
    "FAN-OUT",
    "GATHER-SCATTER",
    "RANDOM",
    "SCATTER-GATHER",
    "STACK"
  ],
  "rows_kept": 179702079,
  "ts_min": "2022-08-01 00:00:00",
  "ts_max": "2023-01-12 16:49:00",
  "laundering_rows": 225546,
  "self_transfers": 10895211,
  "rows_duplicate_removed": 150,
  "duplicate_groups": 150,
  "nodes": 2116168,
  "nodes_without_account_row": 0,
  "rows_raw": 179702229
}
duplicates.parquet           0.0 MB
kyc.parquet                 29.8 MB
patterns.parquet             1.3 MB
transactions.parquet      4799.0 MB
summary.json                 0.0 MB
